In [ ]:
"""
M2
U-NET SEGMENTATION — CALCIFICATIONS VASCULAIRES
Dataset : VinBigData Chest X-ray (Kaggle)
Objectif : Segmenter les zones de calcifications et aortic enlargement
À coller dans un Kaggle Notebook
"""
import os
import pydicom
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import cv2
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import json
import matplotlib.pyplot as plt

# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH     = '/kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection'
TRAIN_CSV     = f'{BASE_PATH}/train.csv'
TRAIN_DIR     = f'{BASE_PATH}/train'
IMG_SIZE      = 512
EPOCHS        = 20
LR            = 1e-4
BATCH         = 8

# Classes cibles pour les calcifications vasculaires
TARGET_CLASSES = ['Calcification', 'Aortic enlargement']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

# ============================================================
# 1. U-NET (même architecture que Phase 2)
# ============================================================
class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super(UNet, self).__init__()

        def double_conv(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c), nn.ReLU(inplace=True)
            )

        self.enc1 = double_conv(in_channels, 64)
        self.enc2 = double_conv(64, 128)
        self.enc3 = double_conv(128, 256)
        self.enc4 = double_conv(256, 512)
        self.pool = nn.MaxPool2d(2, 2)
        self.bottleneck = double_conv(512, 1024)
        self.up4  = nn.ConvTranspose2d(1024, 512, 2, 2)
        self.dec4 = double_conv(1024, 512)
        self.up3  = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec3 = double_conv(512, 256)
        self.up2  = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec2 = double_conv(256, 128)
        self.up1  = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec1 = double_conv(128, 64)
        self.final = nn.Conv2d(64, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([e4, self.up4(b)],  dim=1))
        d3 = self.dec3(torch.cat([e3, self.up3(d4)], dim=1))
        d2 = self.dec2(torch.cat([e2, self.up2(d3)], dim=1))
        d1 = self.dec1(torch.cat([e1, self.up1(d2)], dim=1))
        return torch.sigmoid(self.final(d1))

# ============================================================
# 2. LOSS ET MÉTRIQUES
# ============================================================
class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
        self.bce    = nn.BCELoss()

    def forward(self, pred, target):
        bce   = self.bce(pred, target)
        p, t  = pred.view(-1), target.view(-1)
        inter = (p * t).sum()
        dice  = 1 - (2 * inter + self.smooth) / (p.sum() + t.sum() + self.smooth)
        return bce + dice

def iou_score(pred, target, thr=0.5):
    pb    = (pred > thr).float()
    inter = (pb * target).sum()
    union = pb.sum() + target.sum() - inter
    return (inter + 1e-6) / (union + 1e-6)

def dice_score(pred, target, thr=0.5):
    pb    = (pred > thr).float()
    inter = (pb * target).sum()
    return (2 * inter + 1e-6) / (pb.sum() + target.sum() + 1e-6)

# ============================================================
# 3. DATASET CORRIGÉ
# ============================================================
class CalcifSegDataset(Dataset):
    """
    Corrections par rapport au code original :
    1. target_class → TARGET_CLASSES (Calcification + Aortic enlargement)
    2. Fusion des annotations multi-radiologues (union des bounding boxes)
    3. Masque et image redimensionnés ensemble (ratio conservé)
    4. Gestion des DICOM corrompus
    5. Transforms séparés image / masque (masque sans normalisation)
    """
    def __init__(self, csv_path, img_dir, target_classes, img_size=512):
        self.img_dir        = img_dir
        self.img_size       = img_size
        self.target_classes = target_classes

        df = pd.read_csv(csv_path)
        print(f"CSV total         : {len(df)} annotations")

        # FIX 1 : Filtrer par les bonnes classes
        df_target = df[
            df['class_name'].isin(target_classes) &
            df['x_min'].notnull()
        ].copy()

        self.image_ids = df_target['image_id'].unique()
        self.df        = df_target

        print(f"Classes ciblées   : {target_classes}")
        print(f"Images disponibles: {len(self.image_ids)}")
        for cls in target_classes:
            n = df_target[df_target['class_name'] == cls]['image_id'].nunique()
            print(f"  {cls:<25}: {n} images")

        # FIX 5 : Transforms séparés
        self.img_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])   # normalisation image seulement
        ])
        self.mask_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor()                 # PAS de normalisation sur le masque
        ])

    def __len__(self):
        return len(self.image_ids)

    def _lire_dicom(self, img_id):
        """FIX 4 : Gestion robuste des DICOM corrompus."""
        path = os.path.join(self.img_dir, f"{img_id}.dicom")
        try:
            ds  = pydicom.dcmread(path)
            arr = ds.pixel_array.astype(np.float32)
            arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-6) * 255
            if getattr(ds, 'PhotometricInterpretation', '') == 'MONOCHROME1':
                arr = 255 - arr
            return arr.astype(np.uint8), arr.shape  # (h, w)
        except Exception:
            # Retourner image noire si DICOM corrompu
            return np.zeros((512, 512), dtype=np.uint8), (512, 512)

    def _construire_masque(self, img_id, h, w):
        """
        FIX 2 : Fusion des bounding boxes de tous les radiologues.
        Union de toutes les boxes annotées (OR logique).
        """
        mask  = np.zeros((h, w), dtype=np.uint8)
        boxes = self.df[self.df['image_id'] == img_id][
            ['x_min', 'y_min', 'x_max', 'y_max']
        ].values

        for box in boxes:
            x1, y1, x2, y2 = map(int, box)
            # Clipper dans les limites de l'image
            x1 = max(0, min(x1, w - 1))
            y1 = max(0, min(y1, h - 1))
            x2 = max(0, min(x2, w))
            y2 = max(0, min(y2, h))
            if x2 > x1 and y2 > y1:
                cv2.rectangle(mask, (x1, y1), (x2, y2), 255, -1)

        return mask

    def __getitem__(self, idx):
        img_id          = self.image_ids[idx]
        arr, (h, w)     = self._lire_dicom(img_id)
        mask            = self._construire_masque(img_id, h, w)

        # FIX 3 : Convertir en PIL pour que Resize s'applique identiquement
        img_pil  = Image.fromarray(arr).convert('L')
        mask_pil = Image.fromarray(mask).convert('L')

        # FIX 5 : Appliquer les transforms séparément
        img_tensor  = self.img_transform(img_pil)
        mask_tensor = self.mask_transform(mask_pil)

        # Binariser le masque (0 ou 1)
        mask_tensor = (mask_tensor > 0.5).float()

        return img_tensor, mask_tensor

# ============================================================
# 4. ENTRAÎNEMENT
# ============================================================
def train_unet_calcif():

    dataset = CalcifSegDataset(
        csv_path      = TRAIN_CSV,
        img_dir       = TRAIN_DIR,
        target_classes= TARGET_CLASSES,
        img_size      = IMG_SIZE
    )

    # Split 80/20
    n_train = int(0.80 * len(dataset))
    n_val   = len(dataset) - n_train
    train_set, val_set = random_split(
        dataset, [n_train, n_val],
        generator=torch.Generator().manual_seed(42)
    )
    print(f"\nSplit : Train={n_train} | Val={n_val}")

    train_loader = DataLoader(train_set, batch_size=BATCH,
                              shuffle=True,  num_workers=2)
    val_loader   = DataLoader(val_set,   batch_size=BATCH,
                              shuffle=False, num_workers=2)

    model     = UNet().to(device)
    criterion = DiceBCELoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=4, factor=0.5
    )

    best_val_loss = float('inf')
    historique    = []

    print(f"\n{'Époque':<8} {'Train Loss':<14} {'Val Loss':<12} {'IoU':<10} {'Dice'}")
    print("-" * 56)

    for epoch in range(EPOCHS):
        # TRAIN
        model.train()
        t_loss = 0
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, masks)
            loss.backward()
            optimizer.step()
            t_loss += loss.item()

        # VALIDATION
        model.eval()
        v_loss, v_iou, v_dice = 0, 0, 0
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                out     = model(imgs)
                v_loss += criterion(out, masks).item()
                v_iou  += iou_score(out, masks).item()
                v_dice += dice_score(out, masks).item()

        tl = t_loss / len(train_loader)
        vl = v_loss / len(val_loader)
        vi = v_iou  / len(val_loader)
        vd = v_dice / len(val_loader)

        scheduler.step(vl)
        historique.append({
            'epoch': epoch + 1,
            'train_loss': round(tl, 4),
            'val_loss':   round(vl, 4),
            'iou':        round(vi, 4),
            'dice':       round(vd, 4)
        })

        marker = "  ← meilleur" if vl < best_val_loss else ""
        print(f"{epoch+1:<8} {tl:<14.4f} {vl:<12.4f} {vi:<10.3f} {vd:.3f}{marker}")

        if vl < best_val_loss:
            best_val_loss = vl
            torch.save(model.state_dict(), 'unet_calcif_best.pth')

    torch.save(model.state_dict(), 'unet_calcif_final.pth')
    with open('historique_unet_calcif.json', 'w') as f:
        json.dump(historique, f, indent=2)

    print(f"\nMeilleure Val Loss : {best_val_loss:.4f}")

    # Courbes
    eps  = [x['epoch']      for x in historique]
    tlss = [x['train_loss'] for x in historique]
    vlss = [x['val_loss']   for x in historique]
    ious = [x['iou']        for x in historique]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].plot(eps, tlss, 'b-o', markersize=3, label='Train Loss')
    axes[0].plot(eps, vlss, 'r-o', markersize=3, label='Val Loss')
    axes[0].set_title('Loss — U-Net Calcifications', fontweight='bold')
    axes[0].set_xlabel('Époques')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(eps, ious, 'g-o', markersize=3, label='IoU Val')
    axes[1].set_title('IoU — U-Net Calcifications', fontweight='bold')
    axes[1].set_xlabel('Époques')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('courbes_unet_calcif.png', dpi=150, bbox_inches='tight')
    plt.show()

    return model

# ============================================================
# 5. VISUALISATION D'UN RÉSULTAT
# ============================================================
def visualiser_prediction(model, dataset, idx=0):
    """Affiche image originale + masque réel + masque prédit."""
    model.eval()
    img, mask = dataset[idx]
    input_t   = img.unsqueeze(0).to(device)

    with torch.no_grad():
        pred = model(input_t).cpu().squeeze().numpy()

    img_np  = img.squeeze().numpy()
    mask_np = mask.squeeze().numpy()
    pred_np = (pred > 0.5).astype(np.uint8)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(img_np, cmap='gray')
    axes[0].set_title('Radio originale', fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(img_np, cmap='gray')
    axes[1].imshow(mask_np, cmap='Reds', alpha=0.4)
    axes[1].set_title('Masque réel (bounding boxes)', fontweight='bold')
    axes[1].axis('off')

    axes[2].imshow(img_np, cmap='gray')
    axes[2].imshow(pred_np, cmap='Reds', alpha=0.4)
    axes[2].set_title('Masque prédit (U-Net)', fontweight='bold')
    axes[2].axis('off')

    plt.suptitle('Segmentation Calcifications Vasculaires — U-Net VinDr',
                 fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'prediction_calcif_{idx}.png', dpi=150, bbox_inches='tight')
    plt.show()

# ============================================================
# EXÉCUTION
# ============================================================
if __name__ == "__main__":

    # 1. Entraînement
    model = train_unet_calcif()

    # 2. Visualiser quelques prédictions
    dataset_viz = CalcifSegDataset(
        csv_path=TRAIN_CSV, img_dir=TRAIN_DIR,
        target_classes=TARGET_CLASSES, img_size=IMG_SIZE
    )
    for i in range(3):
        visualiser_prediction(model, dataset_viz, idx=i)

    print("\nFichiers générés :")
    print("  unet_calcif_best.pth      ← à télécharger sur Drive")
    print("  unet_calcif_final.pth")
    print("  historique_unet_calcif.json")
    print("  courbes_unet_calcif.png")